# CardioCare · 01 EDA & 전처리

UCI Heart Disease 통합 데이터셋(4개 processed 파일)을 탐색하고, 전처리 파이프라인 설계 근거를 정리합니다.

In [7]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data_loader import binarize_target, load_raw_data
from src.preprocessing import (
    CONTINUOUS_FEATURES,
    FEATURE_COLUMNS,
    build_preprocessing_pipeline,
)

sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'sklearn'

In [1]:
raw = load_raw_data(ROOT / "data")
df = binarize_target(raw)

print("head():")
display(df[FEATURE_COLUMNS + ["target", "source"]].head())
print("\ninfo():")
df[FEATURE_COLUMNS + ["target"]].info()
print("\ndescribe():")
display(df[FEATURE_COLUMNS].describe())

NameError: name 'load_raw_data' is not defined

In [ ]:
target_dist = df["target"].value_counts(normalize=True)
print("타깃 클래스 분포 (normalize=True):")
print(target_dist)

target_dist.plot(kind="bar", title="Target distribution")
plt.ylabel("proportion")
plt.show()

**클래스 불균형 해석:** 질환(1) 비율이 정상(0)보다 낮으면 accuracy만 보면 다수 클래스 예측으로도 높은 점수가 나올 수 있습니다. 따라서 본 프로젝트에서는 **balanced accuracy, recall, F1**을 함께 사용하고, False Negative(질환을 정상으로 오분류)를 줄이는 방향으로 모델을 선택합니다.

In [ ]:
feature_df = df[FEATURE_COLUMNS]
missing_by_col = feature_df.isna().sum()
missing_total = feature_df.isna().sum().sum()
print("열별 결측값:")
print(missing_by_col[missing_by_col > 0])
print(f"\n전체 결측값 개수: {missing_total}")

In [ ]:
fig, axes = plt.subplots(1, len(CONTINUOUS_FEATURES), figsize=(16, 4))
for ax, col in zip(axes, CONTINUOUS_FEATURES):
    sns.boxplot(y=feature_df[col].dropna(), ax=ax)
    ax.set_title(col)
plt.suptitle("연속형 특성 boxplot (IQR 이상치 탐색)")
plt.tight_layout()
plt.show()

In [ ]:
pipeline = build_preprocessing_pipeline()
X = feature_df
y = df["target"]
X_train = X.sample(frac=0.8, random_state=42)
pipeline.fit(X_train)
X_clean = pipeline.transform(X)

print("전처리 후 shape:", X_clean.shape)
print("중복 행 수 (전처리 전):", X.duplicated().sum())

## EDA가 무엇을 알려 주었으며, 그에 따라 전처리에서 무엇을 어떻게 바꾸었는가?

1. **클래스 불균형:** 질환 양성 비율이 상대적으로 낮아, 이후 모델/평가에서 `class_weight='balanced'`와 recall 중심 지표를 사용했습니다.
2. **결측값:** `-9` 마커 및 일부 컬럼 결측이 확인되어, **컬럼 삭제 대신 중앙값 대치**를 선택했습니다. (샘플 수가 크지 않아 KNN 대치는 과적합 위험)
3. **이상치:** `chol`, `trestbps`, `oldpeak` 등에서 boxplot 꼬리가 길어 **IQR 기반 클리pping**을 적용했습니다. 행 삭제는 정보 손실이 커서 사용하지 않았습니다.
4. **빈 컬럼/중복:** 결측만 있는 컬럼은 제거하고, 중복 레코드는 학습 전 `prepare_dataset()` 단계에서 제거했습니다.
5. **누수 방지:** 스케일링·특성 선택·IQR 경계는 모두 **train split 이후 fit** 하도록 `src/train.py` 파이프라인에 분리했습니다.